In [ ]:

import cv2
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

print(" OpenCV version :", cv2.__version__)
print(" NumPy version  :", np.__version__)
print(" All libraries loaded successfully!")

In [ ]:
import cv2
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

DATASET_PATH = "/kaggle/input/datasets/warcoder/gkn-blade-surface-defect-dataset/GKN Blade Surface" \
" Defect Dataset/Data_GKN"

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

all_files = sorted(glob.glob(DATASET_PATH + "/**/*", recursive=True))
files = [f for f in all_files if f.lower().endswith(IMAGE_EXTENSIONS)]

print(" Dataset Structure:")
print("-" * 50)
folder_counts = {}
for f in files:
    folder = os.path.basename(os.path.dirname(f))
    folder_counts[folder] = folder_counts.get(folder, 0) + 1

for folder, count in sorted(folder_counts.items()):
    bar = "*" * min(count // 5, 20)
    print(f"   {folder:<15} → {count:>4} images  {bar}")

print("-" * 50)
print(f" Total images found: {len(files)}")
print(f"\n   {list(folder_counts.keys())}")
print(f"  {files[0] if files else '...'}")

In [ ]:

categories = ['Good', 'Nick', 'Scratch']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Dataset Preview ', fontsize=14, y=1.02)

for ax, cat in zip(axes, categories):
    cat_files = [f for f in files if f"/{cat}/" in f]
    img = cv2.imread(cat_files[0])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    ax.imshow(img_rgb)
    ax.set_title(f"{cat}\n({len(cat_files)} images)", fontsize=12, fontweight='bold')
    ax.axis('off')

    print(f" {cat}: shape={img.shape} | dtype={img.dtype}")

plt.tight_layout()
plt.show()

In [ ]:

TARGET_SIZE = (512, 512)

def load_and_resize(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
    return img

def show_comparison(original, processed, title_before, title_after, figsize=(12,5)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axes[0].set_title(title_before, fontsize=12, fontweight='bold')
    axes[0].axis('off')

    if len(processed.shape) == 2:  
        axes[1].imshow(processed, cmap='gray')
    else:
        axes[1].imshow(cv2.cvtColor(processed, cv2.COLOR_BGR2RGB))
    axes[1].set_title(title_after, fontsize=12, fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

def show_grid(images_dict, cols=3, figsize=(15, 5)):
    
    items = list(images_dict.items())
    rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if rows > 1 else axes

    for i, (title, img) in enumerate(items):
        if len(img.shape) == 2:
            axes[i].imshow(img, cmap='gray')
        else:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(title, fontsize=10, fontweight='bold')
        axes[i].axis('off')

    for j in range(len(items), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

sample = {}
for cat in ['Good', 'Nick', 'Scratch']:
    cat_files = [f for f in files if f"/{cat}/" in f]
    sample[cat] = load_and_resize(cat_files[0])
    print(f" {cat}: {cv2.imread(cat_files[0]).shape} → {sample[cat].shape}")

print("\n Samples Ready")

In [ ]:

def restore_image(img_bgr):
    
    step1 = cv2.GaussianBlur(img_bgr, (5, 5), 0)

    step2 = cv2.medianBlur(step1, 3)

    restored = cv2.fastNlMeansDenoisingColored(
        step2,
        None,
        h=10,           
        hColor=10,      
        templateWindowSize=7,
        searchWindowSize=21
    )
    return restored

print("  Restoration...")

restored_samples = {}
for cat, img in sample.items():
    restored_samples[cat] = restore_image(img)
    print(f"   {cat} done")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Image Restoration ', fontsize=14, fontweight='bold')

for i, cat in enumerate(['Good', 'Nick', 'Scratch']):
    axes[0][i].imshow(cv2.cvtColor(sample[cat], cv2.COLOR_BGR2RGB))
    axes[0][i].set_title(f'Before — {cat}', fontsize=11)
    axes[0][i].axis('off')

    axes[1][i].imshow(cv2.cvtColor(restored_samples[cat], cv2.COLOR_BGR2RGB))
    axes[1][i].set_title(f'After Restoration — {cat}', fontsize=11)
    axes[1][i].axis('off')

plt.tight_layout()
plt.show()

print("\n Noise Reduction:")
print("-" * 45)
for cat in ['Good', 'Nick', 'Scratch']:
    before = sample[cat].astype(np.float32)
    after  = restored_samples[cat].astype(np.float32)
    diff   = np.mean(np.abs(before - after))
    print(f"  {cat:<10} →  {diff:.4f} pixel")

In [ ]:

def color_correction(img_bgr):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(16, 16))
    l_enhanced = clahe.apply(l)

    lab_merged = cv2.merge([l_enhanced, a, b])
    img_clahe = cv2.cvtColor(lab_merged, cv2.COLOR_LAB2BGR)

    img_float = img_clahe.astype(np.float32)
    avg_b = np.mean(img_float[:, :, 0])
    avg_g = np.mean(img_float[:, :, 1])
    avg_r = np.mean(img_float[:, :, 2])
    avg_gray = (avg_b + avg_g + avg_r) / 3

    img_float[:, :, 0] = np.clip(img_float[:, :, 0] * (avg_gray / avg_b), 0, 255)
    img_float[:, :, 1] = np.clip(img_float[:, :, 1] * (avg_gray / avg_g), 0, 255)
    img_float[:, :, 2] = np.clip(img_float[:, :, 2] * (avg_gray / avg_r), 0, 255)
    img_wb = img_float.astype(np.uint8)

    gamma = 1.2
    table = np.array([((i / 255.0) ** (1.0 / gamma)) * 255
                      for i in range(256)]).astype(np.uint8)
    img_gamma = cv2.LUT(img_wb, table)

    return img_gamma

corrected_samples = {}
for cat, img in restored_samples.items():
    corrected_samples[cat] = color_correction(img)
    print(f"[OK] {cat} - Color Correction done")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Color Correction — Before vs After (Fixed)', 
             fontsize=14, fontweight='bold')

for col, cat in enumerate(['Good', 'Nick', 'Scratch']):
    axes[0][col].imshow(cv2.cvtColor(restored_samples[cat], cv2.COLOR_BGR2RGB))
    axes[0][col].set_title(f'Before — {cat}', fontsize=11)
    axes[0][col].axis('off')

    axes[1][col].imshow(cv2.cvtColor(corrected_samples[cat], cv2.COLOR_BGR2RGB))
    axes[1][col].set_title(f'After Color Correction — {cat}', fontsize=11)
    axes[1][col].axis('off')

plt.tight_layout()
plt.show()

print("\n[INFO] Brightness Report:")
print("-" * 50)
for cat in ['Good', 'Nick', 'Scratch']:
    b_before = np.mean(restored_samples[cat])
    b_after  = np.mean(corrected_samples[cat])
    diff     = b_after - b_before
    arrow    = "UP" if diff > 0 else "DOWN"
    print(f"  {cat:<10} | Before: {b_before:.1f} | After: {b_after:.1f} | {arrow} {abs(diff):.1f}")

print("-" * 50)
print("[DONE] Color Correction complete!")

In [ ]:

def sharpen_image(img_bgr):
    blurred   = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    unsharp   = cv2.addWeighted(img_bgr, 1.5, blurred, -0.5, 0)
    gray      = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    laplacian = np.uint8(np.absolute(laplacian))
    sharpened = cv2.add(gray, laplacian)
    return unsharp, sharpened

unsharp_samples   = {}
laplacian_samples = {}

for cat, img in sample.items():
    u, l = sharpen_image(img)
    unsharp_samples[cat]   = u
    laplacian_samples[cat] = l
    print(f"[OK] {cat} - Sharpening done")

fig, axes = plt.subplots(3, 3, figsize=(15, 14))
fig.suptitle('Image Sharpening — Original vs Unsharp Mask vs Laplacian',
             fontsize=13, fontweight='bold')

for col, cat in enumerate(['Good', 'Nick', 'Scratch']):
    axes[0][col].imshow(cv2.cvtColor(sample[cat], cv2.COLOR_BGR2RGB))
    axes[0][col].set_title(f'Original — {cat}', fontsize=10)
    axes[0][col].axis('off')

    axes[1][col].imshow(cv2.cvtColor(unsharp_samples[cat], cv2.COLOR_BGR2RGB))
    axes[1][col].set_title(f'Unsharp Mask — {cat}', fontsize=10)
    axes[1][col].axis('off')

    axes[2][col].imshow(laplacian_samples[cat], cmap='gray')
    axes[2][col].set_title(f'Laplacian — {cat}', fontsize=10)
    axes[2][col].axis('off')

plt.tight_layout()
plt.show()
print("[DONE] Sharpening complete!")

In [ ]:
import cv2
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

DATASET_PATH = "/kaggle/input/datasets/warcoder/gkn-blade-surface-defect-dataset/GKN Blade Surface Defect Dataset/Data_GKN"
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

all_files = sorted(glob.glob(DATASET_PATH + "/**/*", recursive=True))
files = [f for f in all_files if f.lower().endswith(IMAGE_EXTENSIONS)]
print(f"[OK] Files loaded: {len(files)}")

TARGET_SIZE = (512, 512)
def load_and_resize(img_path):
    img = cv2.imread(img_path)
    return cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)

def restore_image(img_bgr):
    step1 = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    step2 = cv2.medianBlur(step1, 3)
    return cv2.fastNlMeansDenoisingColored(step2, None, h=10, hColor=10,
                                           templateWindowSize=7, searchWindowSize=21)

def color_correction(img_bgr):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(16, 16))
    l_enhanced = clahe.apply(l)
    img_clahe = cv2.cvtColor(cv2.merge([l_enhanced, a, b]), cv2.COLOR_LAB2BGR)
    img_float = img_clahe.astype(np.float32)
    avg_gray = (np.mean(img_float[:,:,0]) + np.mean(img_float[:,:,1]) + np.mean(img_float[:,:,2])) / 3
    for i in range(3):
        img_float[:,:,i] = np.clip(img_float[:,:,i] * (avg_gray / np.mean(img_float[:,:,i])), 0, 255)
    table = np.array([((i/255.0)**(1/1.2))*255 for i in range(256)]).astype(np.uint8)
    return cv2.LUT(img_float.astype(np.uint8), table)

def highlight_damage(img_bgr, sensitivity=1.0):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    _, blade_mask = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    blade_mask = cv2.erode(blade_mask, np.ones((20,20), np.uint8), iterations=1)
    blur_large = cv2.GaussianBlur(gray, (51, 51), 0)
    normalized = np.clip((gray.astype(np.float32) /
                          (blur_large.astype(np.float32)+1e-5))*128, 0, 255).astype(np.uint8)
    normalized[blade_mask == 0] = 0
    morph_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    morph_map = cv2.add(cv2.morphologyEx(normalized, cv2.MORPH_TOPHAT, morph_k),
                        cv2.morphologyEx(normalized, cv2.MORPH_BLACKHAT, morph_k))
    _, morph_mask = cv2.threshold(morph_map, int(30/sensitivity), 255, cv2.THRESH_BINARY)
    k_h = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 1))
    k_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 25))
    k_d = np.eye(15, dtype=np.uint8)
    scratch_map = cv2.add(cv2.add(
        cv2.morphologyEx(normalized, cv2.MORPH_TOPHAT, k_h),
        cv2.morphologyEx(normalized, cv2.MORPH_TOPHAT, k_v)),
        cv2.morphologyEx(normalized, cv2.MORPH_TOPHAT, k_d))
    _, scratch_mask = cv2.threshold(scratch_map, int(28/sensitivity), 255, cv2.THRESH_BINARY)
    combined = cv2.bitwise_and(cv2.bitwise_or(morph_mask, scratch_mask),
                                cv2.bitwise_or(morph_mask, scratch_mask), mask=blade_mask)
    overlay = img_bgr.copy()
    overlay[combined > 0] = [0, 0, 255]
    result = cv2.addWeighted(img_bgr, 0.6, overlay, 0.4, 0)
    return result, combined, cv2.add(morph_map, scratch_map), blade_mask

sample = {}
corrected_samples = {}
for cat in ['Good', 'Nick', 'Scratch']:
    cat_files = [f for f in files if f"/{cat}/" in f]
    img = load_and_resize(cat_files[0])
    sample[cat] = img
    corrected_samples[cat] = color_correction(restore_image(img))
    print(f"[OK] {cat} sample loaded and processed")

print("\n[DONE] All variables reloaded — ready to continue!")

In [ ]:

def sobel_edge_detection(img_bgr):
    
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_x = np.uint8(np.absolute(sobel_x))

    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel_y = np.uint8(np.absolute(sobel_y))

    sobel_combined = cv2.addWeighted(sobel_x, 0.5, sobel_y, 0.5, 0)

    canny = cv2.Canny(gray, 50, 150)

    return sobel_x, sobel_y, sobel_combined, canny

sobel_x_samples   = {}
sobel_y_samples   = {}
sobel_combined    = {}
canny_samples     = {}

for cat, img in sample.items():
    sx, sy, sc, cn = sobel_edge_detection(img)
    sobel_x_samples[cat]  = sx
    sobel_y_samples[cat]  = sy
    sobel_combined[cat]   = sc
    canny_samples[cat]    = cn
    print(f"[OK] {cat} - Sobel done")

fig, axes = plt.subplots(4, 3, figsize=(15, 18))
fig.suptitle('Sobel Edge Detection vs Canny — All Categories',
             fontsize=13, fontweight='bold')

row_labels = ['Sobel X (Vertical Edges)',
              'Sobel Y (Horizontal Edges)',
              'Sobel Combined',
              'Canny (for comparison)']

for col, cat in enumerate(['Good', 'Nick', 'Scratch']):
    axes[0][col].imshow(sobel_x_samples[cat],  cmap='gray')
    axes[0][col].set_title(f'Sobel X — {cat}', fontsize=10)
    axes[0][col].axis('off')

    axes[1][col].imshow(sobel_y_samples[cat],  cmap='gray')
    axes[1][col].set_title(f'Sobel Y — {cat}', fontsize=10)
    axes[1][col].axis('off')

    axes[2][col].imshow(sobel_combined[cat],   cmap='gray')
    axes[2][col].set_title(f'Sobel Combined — {cat}', fontsize=10)
    axes[2][col].axis('off')

    axes[3][col].imshow(canny_samples[cat],    cmap='gray')
    axes[3][col].set_title(f'Canny — {cat}',  fontsize=10)
    axes[3][col].axis('off')

plt.tight_layout()
plt.show()

print("\n[INFO] Edge Intensity Report (mean pixel value):")
print("-" * 55)
print(f"  {'Category':<10} | {'Sobel X':>8} | {'Sobel Y':>8} | {'Combined':>9} | {'Canny':>7}")
print("-" * 55)
for cat in ['Good', 'Nick', 'Scratch']:
    print(f"  {cat:<10} | "
          f"{np.mean(sobel_x_samples[cat]):>8.2f} | "
          f"{np.mean(sobel_y_samples[cat]):>8.2f} | "
          f"{np.mean(sobel_combined[cat]):>9.2f} | "
          f"{np.mean(canny_samples[cat]):>7.2f}")
print("-" * 55)
print("[DONE] Sobel Edge Detection complete!")

In [ ]:

def simple_segmentation(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

    _, otsu = cv2.threshold(gray, 0, 255,
                             cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    adaptive = cv2.adaptiveThreshold(gray, 255,
                                      cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY, 11, 2)
    return binary, otsu, adaptive

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
fig.suptitle('Segmentation — Binary vs Otsu vs Adaptive',
             fontsize=13, fontweight='bold')

for col, cat in enumerate(['Good', 'Nick', 'Scratch']):
    b, o, a = simple_segmentation(sample[cat])

    axes[0][col].imshow(b, cmap='gray')
    axes[0][col].set_title(f'Binary Threshold — {cat}', fontsize=10)
    axes[0][col].axis('off')

    axes[1][col].imshow(o, cmap='gray')
    axes[1][col].set_title(f'Otsu Threshold — {cat}', fontsize=10)
    axes[1][col].axis('off')

    axes[2][col].imshow(a, cmap='gray')
    axes[2][col].set_title(f'Adaptive Threshold — {cat}', fontsize=10)
    axes[2][col].axis('off')

plt.tight_layout()
plt.show()
print("[DONE] Segmentation complete!")

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(15, 22))
fig.suptitle('Complete Pipeline — All Techniques per Category',
             fontsize=14, fontweight='bold')

for col, cat in enumerate(['Good', 'Nick', 'Scratch']):

    axes[0][col].imshow(cv2.cvtColor(sample[cat], cv2.COLOR_BGR2RGB))
    axes[0][col].set_title(f'1. Original — {cat}', fontsize=10)
    axes[0][col].axis('off')

    restored = restore_image(sample[cat])
    axes[1][col].imshow(cv2.cvtColor(restored, cv2.COLOR_BGR2RGB))
    axes[1][col].set_title(f'2. Restored (NLM) — {cat}', fontsize=10)
    axes[1][col].axis('off')

    axes[2][col].imshow(cv2.cvtColor(corrected_samples[cat], cv2.COLOR_BGR2RGB))
    axes[2][col].set_title(f'3. Color Corrected — {cat}', fontsize=10)
    axes[2][col].axis('off')

    axes[3][col].imshow(cv2.cvtColor(unsharp_samples[cat], cv2.COLOR_BGR2RGB))
    axes[3][col].set_title(f'4. Sharpened — {cat}', fontsize=10)
    axes[3][col].axis('off')

    axes[4][col].imshow(sobel_combined[cat], cmap='gray')
    axes[4][col].set_title(f'5. Sobel Edges — {cat}', fontsize=10)
    axes[4][col].axis('off')

plt.tight_layout()
plt.show()

print("[INFO] Complete Pipeline Summary:")
print("=" * 55)
print(f"  {'Stage':<25} | {'Technique':<26}")
print("=" * 55)
stages = [
    ("1. Dataset Collection",  "glob + cv2.imread + BGR filter"),
    ("2. Resize",              "cv2.resize — INTER_AREA 512x512"),
    ("3. Restoration",         "Gaussian + Median + Non-Local Means"),
    ("4. Color Correction",    "CLAHE + White Balance + Gamma"),
    ("5. Sharpening",          "Unsharp Mask + Laplacian"),
    ("6. Edge Detection",      "Sobel X/Y + Canny"),
    ("7. Segmentation",        "Binary + Otsu + Adaptive Threshold"),
]
for stage, tech in stages:
    print(f"  {stage:<25} | {tech:<26}")
print("=" * 55)
print(f"  Total images : 400 | Good:203 | Nick:48 | Scratch:149")
print("=" * 55)
print("[DONE] Pipeline complete!")